In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2" 

import sys
sys.path.append("/home/cyf/wbi/Virginia/code")

import json
import torch
from importlib import reload

from CoarseFlow.training import train, losses
from CoarseFlow.datasets import synthetic_dataset

reload(train)
reload(losses)
reload(synthetic_dataset)

from CoarseFlow.training.train import train_coarse_matching_model
from CoarseFlow.datasets.synthetic_dataset import (
    build_sameShape_loader,
    summarize_manifest,
)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("current device:", torch.cuda.current_device())
    print("device name:", torch.cuda.get_device_name(0))

torch: 2.6.0+cu124
cuda available: True
device count: 1
current device: 0
device name: NVIDIA A100-SXM4-80GB


读取manifest_summary.json
---

In [2]:
# train.ipynb - Cell 2

MANIFEST_SUMMARY_PATH = "/home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_v6/manifest_summary.json"

with open(MANIFEST_SUMMARY_PATH, "r") as f:
    manifest_summary = json.load(f)

for k, v in manifest_summary.items():
    print(k, "->", v)

for name, path in manifest_summary.items():
    print("\n" + "=" * 100)
    print(name)
    summarize_manifest(path)

stage0_train -> /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_v6/stage0_sanity_train/stage0_train_manifest.json
stage0_val -> /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_v6/stage0_sanity_val/stage0_val_manifest.json
stage1_train -> /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_v6/stage1_K5_varSpacing_train/stage1_train_manifest.json
stage1_val -> /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_v6/stage1_K5_varSpacing_val/stage1_val_manifest.json
stage2_train -> /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_v6/stage2_varK_varSpacing_train/stage2_train_manifest.json
stage2_val -> /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_v6/stage2_varK_varSpacing_val/stage2_val_manifest.json
stage3_train -> /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_v6/stage3_hard_train/stage3_train_manifest.json
stage3_val -> /home/cyf/wbi/Virginia/code/CoarseFlow/cached_data

定义模型
---


In [ ]:
model_config_v6 = dict(
    # =====================================================
    # Core
    # =====================================================
    dim=96,
    radius=(4, 3, 3),
    temperature=0.05,
    use_learned_matching=True,
    matcher_mode="hybrid",

    # feature stride = /8, output control grid = /16
    control_stride=16,
    encoder_stride=8,

    # Important: use 1 refinement iterations from the beginning
    num_refine_iters=1,
    query_chunk_size=512,

    # =====================================================
    # Moving encoder: raw moving -> /8 feature
    # =====================================================
    moving_base_channels=(24, 48, 96),
    moving_num_blocks=(4, 8, 8),
    moving_mlp_ratio=2.0,

    # Important: align moving attention depth with reference attention
    moving_window_attn_layers=6,
    moving_window_size=8,
    moving_attn_num_heads=4,
    moving_slice_fusion_blocks=1,

    # =====================================================
    # Reference encoder: raw ref -> /8 feature
    # =====================================================
    ref_base_channels=(24, 48, 96),
    ref_num_blocks=(4, 8, 8),
    ref_refine_blocks=1,
    ref_mlp_ratio=2.0,

    # Important: 6-layer 3D reference attention
    ref_attn_layers=6,
    ref_attn_num_heads=4,
    ref_attn_window_size=(4, 8, 8),
    ref_attn_mlp_ratio=2.0,

    # =====================================================
    # Embeddings
    # =====================================================
    use_coord_embed=True,
    use_spacing_embed=True,

    # =====================================================
    # Matcher V2
    # =====================================================
    use_offset_encoding=True,
    use_offset_bias=True,
    use_local_cross_attn=True,
    local_attn_temperature=0.20,
    matcher_cross_attn_layers=3,
    matcher_cross_attn_heads=4,
    matcher_ffn_ratio=2.0,
    matcher_attn_drop=0.0,
    matcher_proj_drop=0.0,
    matcher_init_gamma=1e-2,
    
    # =====================================================
    # Residual refinement: OFF for Stage 1-3
    # =====================================================
    use_coord_residual=False,
    residual_type="spatial",
    residual_hidden_dim=256,
    residual_num_blocks=5,
    residual_max_delta=(1.5, 3.0, 3.0),
    residual_use_disp=True,
    residual_use_3d=True,
    residual_detach_coarse=True,
    residual_detach_features=True,
)

In [ ]:
# ============================================================
# Stage 1: K=5 + variable spacing
# ============================================================

train_loader_stage1, _, _ = build_sameShape_loader(
    manifest_summary["stage1_train"],
    batch_size=2,   # 如果显存够，可以改回 5
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

val_loader_stage1, _, _ = build_sameShape_loader(
    manifest_summary["stage1_val"],
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

model_config_stage1 = dict(model_config_v6)

model_stage1 = train_coarse_matching_model(
    train_dataset=None,
    val_dataset=None,
    train_loader=train_loader_stage1,
    val_loader=val_loader_stage1,

    save_dir="./checkpoints/coarseflow_v7_stage1_K5_varSpacing_iter3_attn6",

    num_epochs=150,
    lr=5e-5,
    weight_decay=1e-4,
    batch_size=2,
    num_workers=0,
    use_amp=True,

    **model_config_stage1,

    loss_mode="match",

    # 初期仍然以 match distribution 为主
    lambda_match=1.0,
    lambda_match_kl=0.0,
    lambda_match_ce=1.0,

    # 有 3 次 refine 后，coord 可以稍微保留
    lambda_coord=0.1,
    lambda_disp=0.0,

    # 前期不要太强 regularization
    lambda_smooth=0.001,
    lambda_z_spacing=0.001,
    lambda_disp_mag=0.5,

    compute_chunk_match_loss=True,
    match_sigma=(0.5, 1.0, 1.0),
    match_inside_threshold=4.0,

    resume_path=None,
    resume_optimizer=False,
    resume_best_val_loss=False,
    strict_load=False,

    log_filename="train.log",
    log_mode="w",
)

[SameShapeBatchSampler] groups:
  key=(K,D,H,W)=(5, 20, 256, 256), n=339
  key=(K,D,H,W)=(5, 30, 256, 256), n=111
[SameShapeBatchSampler] groups:
  key=(K,D,H,W)=(5, 20, 256, 256), n=71
  key=(K,D,H,W)=(5, 30, 256, 256), n=19


KeyboardInterrupt: 

In [ ]:
# ============================================================
# Stage 2: variable K + variable spacing
# ============================================================

train_loader_stage2, _, _ = build_sameShape_loader(
    manifest_summary["stage2_train"],
    batch_size=2,   # 如果显存够，可以改成 4
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

val_loader_stage2, _, _ = build_sameShape_loader(
    manifest_summary["stage2_val"],
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

model_config_stage2 = dict(model_config_v6)


model_stage2 = train_coarse_matching_model(
    train_dataset=None,
    val_dataset=None,
    train_loader=train_loader_stage2,
    val_loader=val_loader_stage2,

    save_dir="./checkpoints/coarseflow_v7_stage2_varK_varSpacing_iter3_attn6",

    num_epochs=300,
    lr=3e-5,
    weight_decay=1e-4,
    batch_size=3,
    num_workers=0,
    use_amp=True,

    **model_config_stage2,

    loss_mode="match",

    lambda_match=1.0,
    lambda_match_kl=0.9,
    lambda_match_ce=0.4,

    lambda_coord=0.1,
    lambda_disp=0.0,
    lambda_smooth=0.001,
    lambda_z_spacing=0.001,

    compute_chunk_match_loss=True,
    match_sigma=(0.5, 1.0, 1.0),
    match_inside_threshold=4.0,

    resume_path="./checkpoints/coarseflow_v7_stage1_K5_varSpacing_iter3_attn6/best.pth",
    resume_optimizer=False,
    resume_best_val_loss=False,
    strict_load=True,

    log_filename="train.log",
    log_mode="w",
)

[SameShapeBatchSampler] groups:
  key=(K,D,H,W)=(3, 20, 256, 256), n=230
  key=(K,D,H,W)=(3, 30, 256, 256), n=70
  key=(K,D,H,W)=(4, 20, 256, 256), n=235
  key=(K,D,H,W)=(4, 30, 256, 256), n=65
  key=(K,D,H,W)=(5, 20, 256, 256), n=217
  key=(K,D,H,W)=(5, 30, 256, 256), n=83
  key=(K,D,H,W)=(6, 20, 256, 256), n=223
  key=(K,D,H,W)=(6, 30, 256, 256), n=77
  key=(K,D,H,W)=(7, 20, 256, 256), n=224
  key=(K,D,H,W)=(7, 30, 256, 256), n=76
[SameShapeBatchSampler] groups:
  key=(K,D,H,W)=(3, 20, 256, 256), n=51
  key=(K,D,H,W)=(3, 30, 256, 256), n=21
  key=(K,D,H,W)=(5, 20, 256, 256), n=60
  key=(K,D,H,W)=(5, 30, 256, 256), n=12
  key=(K,D,H,W)=(7, 20, 256, 256), n=62
  key=(K,D,H,W)=(7, 30, 256, 256), n=10
Total parameters    : 1.885 M
Trainable parameters: 1.885 M
Frozen parameters   : 0.000 M
Module                                       Total(M)    Trainable(M)
--------------------------------------------------------------------------------
coord_mlp                                       0.

/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:814: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=use_amp)
/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:161: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


2026-05-20 10:13:16 | [Epoch 095] step 0000/505 | loss=2.5330 | match=2.0916 | kl=1.0670 | ce=2.8284 | coord=4.3710 | disp=4.3710 | smooth=1.2946 | z_spacing=2.9717 | top1=0.426 | top5=0.764 | pmax=0.170 | ent=3.761 | inside_valid=1.000 | cand_valid=0.765 | inside_all=0.972 | valid_inside_all=0.551 | time=2.3s | valid=0.551 | 
2026-05-20 10:13:30 | [Epoch 095] step 0010/505 | loss=1.8698 | match=1.5554 | kl=0.7030 | ce=2.3068 | coord=3.1176 | disp=3.1176 | smooth=1.1917 | z_spacing=1.4954 | top1=0.341 | top5=0.963 | pmax=0.193 | ent=3.622 | inside_valid=1.000 | cand_valid=0.763 | inside_all=1.000 | valid_inside_all=0.707 | time=15.6s | valid=0.707 | 
2026-05-20 10:13:42 | [Epoch 095] step 0020/505 | loss=1.9276 | match=1.6441 | kl=0.7770 | ce=2.3620 | coord=2.8139 | disp=2.8139 | smooth=1.0484 | z_spacing=1.0317 | top1=0.325 | top5=0.941 | pmax=0.201 | ent=3.580 | inside_valid=1.000 | cand_valid=0.769 | inside_all=1.000 | valid_inside_all=0.594 | time=27.9s | valid=0.594 | 
2026-05-20 

/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:340: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


2026-05-20 10:24:00 | [Val Epoch 095] loss=2.2236 | match=1.8470 | kl=0.8445 | ce=2.7175 | coord=3.7378 | disp=3.7378 | smooth=1.2293 | z_spacing=1.5942 | top1=0.390 | top5=0.854 | pmax=0.143 | ent=3.946 | inside_valid=1.000 | cand_valid=0.780 | inside_all=0.984 | valid_inside_all=0.608 | valid=0.608 | pred_mag=1.321 | gt_mag=2.582
2026-05-20 10:24:01 | [Epoch 096] step 0000/505 | loss=1.9928 | match=1.6767 | kl=0.6962 | ce=2.6252 | coord=3.1473 | disp=3.1473 | smooth=1.2920 | z_spacing=0.1082 | top1=0.242 | top5=0.987 | pmax=0.137 | ent=3.986 | inside_valid=1.000 | cand_valid=0.830 | inside_all=1.000 | valid_inside_all=0.619 | time=0.8s | valid=0.619 | 
2026-05-20 10:24:14 | [Epoch 096] step 0010/505 | loss=2.7765 | match=2.3434 | kl=1.2063 | ce=3.1443 | coord=4.3072 | disp=4.3072 | smooth=1.2596 | z_spacing=1.1422 | top1=0.042 | top5=0.693 | pmax=0.189 | ent=3.717 | inside_valid=1.000 | cand_valid=0.782 | inside_all=0.971 | valid_inside_all=0.636 | time=13.7s | valid=0.636 | 
2026-05

In [2]:
# ============================================================
# Stage 3: sharpen / fine-tune with iter3
# ============================================================

train_loader_stage3, _, _ = build_sameShape_loader(
    manifest_summary["stage3_train"],
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

val_loader_stage3, _, _ = build_sameShape_loader(
    manifest_summary["stage3_val"],
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

model_config_stage3 = dict(model_config_v6)
model_config_stage3.update(
    dict(
        num_refine_iters=3,
        moving_window_attn_layers=6,
        ref_attn_layers=6,
        use_coord_residual=False,
    )
)

model_stage3 = train_coarse_matching_model(
    train_dataset=None,
    val_dataset=None,
    train_loader=train_loader_stage3,
    val_loader=val_loader_stage3,

    save_dir="./checkpoints/coarseflow_v6_stage3_iter3_attn6_sharpen",

    num_epochs=120,
    lr=5e-6,
    weight_decay=1e-4,
    batch_size=2,
    num_workers=0,
    use_amp=True,

    **model_config_stage3,

    loss_mode="match",

    # Sharpen distribution
    lambda_match=1.0,
    lambda_match_kl=0.25,
    lambda_match_ce=0.45,

    # More coordinate supervision
    lambda_coord=0.3,
    lambda_disp=0.0,

    lambda_smooth=0.001,
    lambda_z_spacing=0.001,

    compute_chunk_match_loss=True,
    match_sigma=(0.5, 1.0, 1.0),
    match_inside_threshold=4.0,

    resume_path="./checkpoints/coarseflow_v7_stage2_varK_varSpacing_iter3_attn6/best.pth",
    resume_optimizer=False,
    resume_best_val_loss=False,
    strict_load=True,

    log_filename="train.log",
    log_mode="w",
)

NameError: name 'manifest_summary' is not defined